# Demo Pipeline — point de départ

**Objectif du challenge** (extrait de `PARTICIPANT_PACK/READ_ME.md`) : *concevoir une solution capable de
renseigner des questionnaires inconnus à partir du contexte de l'entreprise assignée, en justifiant chaque
réponse et en signalant les informations insuffisantes.*

Ce notebook montre **comment accéder aux données** du pack participant et **quel livrable produire**.
Il ne contient **aucune solution, aucune réponse et aucune méthode imposée** : à vous de concevoir votre
pipeline. Toutes les règles citées ici proviennent de `READ_ME.md`, qui reste la référence.

In [ ]:
import csv
import json
import re
from pathlib import Path

# Le pack participant se trouve à la racine du dépôt, dans PARTICIPANT_PACK/.
DATA = next(p / "PARTICIPANT_PACK" for p in [Path.cwd(), *Path.cwd().parents] if (p / "PARTICIPANT_PACK").is_dir())
print("Données :", DATA.name, "-", sorted(p.name for p in DATA.iterdir()))

## 1. Les cinq exercices

Chaque exercice associe **un questionnaire PDF** et **le contexte d'une seule entreprise** : il y a cinq
questionnaires à traiter, pas quinze combinaisons. `exercices.json` donne la correspondance.

In [ ]:
exercices = json.loads((DATA / "exercices.json").read_text(encoding="utf-8"))
for e in exercices:
    print(f"{e['exercice']}  {e['entreprise']:<28} {e['langue']:<20} {e['questionnaire']}")

## 2. Retrouver les documents d'une entreprise

D'après le READ_ME : commencez par `index_entreprises.json`, puis le `manifest.json` de l'entreprise.
Certains identifiants de source commencent par un préfixe de stockage `A/`, `B/` ou `C/` : remplacez-le par
`source_prefix_local_target` pour retrouver le fichier local. Les chemins du manifest sont relatifs au
dossier de l'entreprise.

In [ ]:
index = json.loads((DATA / "index_entreprises.json").read_text(encoding="utf-8"))


def resolve(source_ref: str) -> Path:
    """Chemin local d'un identifiant de source préfixé (règle du READ_ME)."""
    for company in index.values():
        if source_ref.startswith(company["source_prefix"]):
            return DATA / company["source_prefix_local_target"] / source_ref[len(company["source_prefix"]):]
    return DATA / source_ref


# Exemple donné par le READ_ME : A/sources/... se trouve dans entreprises/asterive_services/sources/...
example = resolve("A/sources/companies/sim-hackathon-a-client/corporate.md")
print(example.relative_to(DATA), "->", example.exists())

In [ ]:
# Manifest de l'entreprise du premier exercice
company_key = Path(exercices[0]["contexte"]).name
company_dir = DATA / index[company_key]["folder"]
manifest = json.loads((company_dir / "manifest.json").read_text(encoding="utf-8"))

print("Clés du manifest   :", list(manifest))
print("Situation au       :", manifest["as_of"])
print("Tables             :", list(manifest["table_files"]))
print("Documents sources  :", len(manifest["source_documents"]))

## 3. Lire les différents formats

Le pack contient :
- des registres **Markdown** : un en-tête rédigé, suivi d'un bloc ```` ```json ```` structuré ;
- des documents **JSON** avec une enveloppe (`document_type`, `declared_at`, `data`) ;
- des fichiers **CSV** ;
- des **tables** JSON (`tables/`) qui suivent l'organisation de la base.

Rappel du READ_ME : *un fait recopié dans plusieurs tables ne représente pas plusieurs preuves
indépendantes*, et il faut vérifier l'entité, le périmètre, la date et le statut de chaque document.

In [ ]:
def read_markdown(path: Path) -> dict:
    """Titre, texte d'en-tête et bloc JSON d'un registre Markdown."""
    text = path.read_text(encoding="utf-8")
    block = re.search(r"```json\s*\n(.*?)```", text, re.S)
    title = re.search(r"^# (.+)$", text, re.M)
    return {"title": title.group(1) if title else None,
            "header": re.sub(r"```json.*?```", "", text, flags=re.S).strip(),
            "data": json.loads(block.group(1)) if block else None}


doc = read_markdown(company_dir / manifest["source_documents"][0])
print("Titre               :", doc["title"])
print("Clés du bloc JSON   :", list(doc["data"]) if isinstance(doc["data"], dict) else type(doc["data"]).__name__)

In [ ]:
def read_json_source(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


person_file = next((company_dir / "sources" / "persons").glob("*/*.json"))
person = read_json_source(person_file)
print("Document personne   :", person["document_type"], "- clés :", list(person["data"]))

table = read_json_source(company_dir / manifest["table_files"]["subsidiaries"])
print("Table subsidiaries  :", len(table), "lignes - colonnes :", list(table[0]))

csv_file = next((company_dir / "sources" / "companies").glob("*/*.csv"))
with csv_file.open(encoding="utf-8", newline="") as fh:
    print("CSV                 :", csv_file.name, "- colonnes :", next(csv.reader(fh)))

## 4. Les questionnaires

Les questionnaires sont des **PDF scannés** : ils ne contiennent ni couche texte ni champs de formulaire.
À vous de choisir comment en extraire les questions et les zones à remplir.

La cellule suivante utilise `pymupdf` (`pip install pymupdf`) ; elle est facultative.

In [ ]:
try:
    import pymupdf
except ImportError:
    pymupdf = None
    print("pymupdf n'est pas installé : cellule ignorée.")

if pymupdf:
    for e in exercices:
        pdf = pymupdf.open(DATA / e["questionnaire"])
        has_text = any(page.get_text().strip() for page in pdf)
        print(f"{e['exercice']}: {pdf.page_count} pages, couche texte : {'oui' if has_text else 'non'}")

In [ ]:
if pymupdf:
    from IPython.display import Image, display

    first_page = pymupdf.open(DATA / exercices[0]["questionnaire"])[0]
    display(Image(first_page.get_pixmap(dpi=50).tobytes("png")))

## 5. Le livrable attendu

Extrait de `READ_ME.md`, section *Livrables* : pour chaque exercice, **un PDF complété** et **un JSON de
réponses** avec, pour chaque champ :

- la page et le libellé du champ (le couple page + libellé permet la correspondance) ;
- la valeur ;
- l'état, parmi `answer`, `not_applicable`, `missing_information`, `bank_reserved`, `human_action` ;
- la source précise (document et section / clé / ligne) ;
- la justification ou le calcul si nécessaire.

Pour une réponse partielle, conservez les éléments connus et listez les composants inconnus.

Rappels des *Règles de réponse* : distinguez « Non », zéro, non applicable et information manquante ;
n'inventez ni donnée ni signature ; laissez les champs réservés à la banque et les signatures non exécutées.

La forme exacte du JSON ci-dessous est **une proposition** qui couvre ces éléments, pas un format imposé.

In [ ]:
STATES = {"answer", "not_applicable", "missing_information", "bank_reserved", "human_action"}


def make_answer(page: int, label: str, value, state: str, source: str | None = None,
                justification: str | None = None, missing: list[str] | None = None) -> dict:
    return {"page": page, "label": label, "value": value, "state": state, "source": source,
            "justification": justification, "missing": missing or []}


# Enregistrements FACTICES : ils montrent la forme, pas le contenu d'un vrai questionnaire.
example_answers = [
    make_answer(1, "<libellé du champ>", "<valeur>", "answer",
                source="<fichier>#<section, clé ou ligne>", justification="<pourquoi cette valeur>"),
    make_answer(1, "<autre libellé>", None, "missing_information",
                source="<fichier>#<clé>", justification="<pourquoi l'information manque>",
                missing=["<composant inconnu>"]),
]
print(json.dumps(example_answers, ensure_ascii=False, indent=2))

In [ ]:
def check_format(answers: list[dict]) -> list[str]:
    """Contrôle de FORME uniquement (champs présents, état autorisé) : il ne juge pas les réponses."""
    problems = []
    for i, a in enumerate(answers):
        for key in ("page", "label", "value", "state", "source"):
            if key not in a:
                problems.append(f"#{i}: champ '{key}' absent")
        if a.get("state") not in STATES:
            problems.append(f"#{i}: état inconnu {a.get('state')!r}")
        if a.get("state") == "missing_information" and not a.get("missing"):
            problems.append(f"#{i}: préciser les composants manquants")
    return problems


print(check_format(example_answers) or "Format OK")

# Pour enregistrer votre propre livrable, par exemple :
# out = Path("submission") / "form_01.answers.json"
# out.parent.mkdir(exist_ok=True)
# out.write_text(json.dumps(my_answers, ensure_ascii=False, indent=2), encoding="utf-8")

## 6. À vous de jouer

À construire librement, avec les outils de votre choix :

1. extraire les questions et les zones de chaque questionnaire ;
2. retrouver dans les sources de l'entreprise assignée les éléments qui y répondent ;
3. décider de la valeur et de l'état de chaque champ, avec sa source et sa justification ;
4. produire le JSON de réponses et le PDF complété.

À remettre aussi (READ_ME, *Livrables*) : votre code et un README permettant de reproduire le résultat
(installation, commande, modèles et dépendances, durée et coûts observés). Le canal de remise, l'horaire,
les outils autorisés et le budget API seront communiqués par les organisateurs.

*Exécution de ce notebook : `pip install jupyter pymupdf`, puis `jupyter lab notebooks/demo_pipeline.ipynb`
depuis la racine du dépôt.*